> **Solución.** Notebook original del curso con las actividades (`TODO`) completadas. Corre de principio a fin (`Restart & Run All`) y las celdas de prueba (`assert`) pasan.
>
> Nicolás Rodríguez

# Optimización local: el problema de las 4 reinas

En este ejercicio modelaremos el problema de las **4 reinas** como un problema de optimización y construiremos tres algoritmos:

1. **Hill Climbing**
2. **Hill Climbing con Random Restart**
3. **Simulated Annealing**

La mayor parte de las funciones están incompletas. Deben implementarse durante la clase.

---

## Objetivo

Ubicar cuatro reinas en un tablero $4 \times 4$ de forma que ninguna pareja se ataque.

Una reina ataca a otra si ambas están:

- en la misma fila;
- en la misma columna;
- en la misma diagonal.

## 1. Formulación como problema de optimización

Representaremos un estado mediante un vector:

$$
x = [x_0, x_1, x_2, x_3]
$$

donde $x_i$ indica la **fila** de la reina ubicada en la columna $i$.

Por ejemplo:

$$
x = [1, 3, 0, 2]
$$

representa las posiciones:

$$
(1,0), (3,1), (0,2), (2,3)
$$

Como cada columna contiene exactamente una reina, no pueden existir conflictos por columna.

### Espacio de búsqueda

Cada una de las cuatro reinas puede estar en cualquiera de las cuatro filas:

$$
|\mathcal{X}| = 4^4 = 256
$$

### Función de costo

Definiremos:

$$
C(x) = \text{número de parejas de reinas que se atacan}
$$

El objetivo es resolver:

$$
\min_{x \in \mathcal{X}} C(x)
$$

Una solución válida satisface:

$$
C(x)=0
$$

In [1]:
import math
import random
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

cmap = ListedColormap(["#F0D9B5", "#B58863"])



N = 4
SEED = 42
random.seed(SEED)

## 2. Visualización de un estado

In [2]:
def plot_board(state, title=None):
    """Visualiza un estado del problema de las N reinas."""
    n = len(state)
    board = [[(row + col) % 2 for col in range(n)] for row in range(n)]

    plt.figure(figsize=(4, 4))
    plt.imshow(board, cmap="Greys", vmin=0, vmax=1)

    for col, row in enumerate(state):
        plt.text(col, row, "♛", ha="center", va="center", fontsize=32)

    plt.xticks(range(n))
    plt.yticks(range(n))
    plt.xlabel("Columna")
    plt.ylabel("Fila")
    plt.grid(False)
    if title:
        plt.title(title)

    plt.imshow(board, cmap=cmap)


example_state = [1, 3, 0, 2]
plot_board(example_state, title=f"Estado {example_state}")

## Actividad 1. Detectar conflictos

Dos reinas ubicadas en $(r_i,c_i)$ y $(r_j,c_j)$ se atacan si:

$$
r_i = r_j
$$

o si están en la misma diagonal:

$$
|r_i-r_j| = |c_i-c_j|
$$

Complete la función `attacking`.

In [3]:
def attacking(row_i, col_i, row_j, col_j):
    """True si dos reinas se atacan: misma fila o misma diagonal
    (siempre están en columnas distintas)."""
    return row_i == row_j or abs(row_i - row_j) == abs(col_i - col_j)

In [4]:
# Pruebas de la actividad 1
assert attacking(0, 0, 0, 3) is True       # misma fila
assert attacking(0, 0, 3, 3) is True       # misma diagonal
assert attacking(0, 0, 1, 3) is False      # no se atacan

print("Pruebas superadas.")

Pruebas superadas.


## Actividad 2. Función de costo

Implemente una función que cuente cuántas parejas de reinas se atacan.

Para cuatro reinas existen:

$$
\binom{4}{2}=6
$$

parejas posibles.

In [5]:
def cost(state):
    """Número de parejas de reinas que se atacan."""
    n = len(state)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if attacking(state[i], i, state[j], j))

In [6]:
# Pruebas de la actividad 2
assert cost([0, 0, 0, 0]) == 6
assert cost([0, 1, 2, 3]) == 6
assert cost([1, 3, 0, 2]) == 0

print("Pruebas superadas.")

Pruebas superadas.


## 3. Vecindario

Un vecino se obtiene moviendo **una sola reina** a otra fila de su misma columna.

Para cada una de las $N$ columnas hay $N-1$ movimientos posibles:

$$
|N(x)| = N(N-1)
$$

Para $N=4$:

$$
|N(x)|=4(3)=12
$$

## Actividad 3. Generar los vecinos

Implemente `neighbors(state)`.

La función debe retornar todos los estados que difieren del original en la fila de exactamente una reina.

In [7]:
def neighbors(state):
    """Todos los vecinos: mover una reina a otra fila dentro de su misma columna."""
    result = []
    n = len(state)
    for col in range(n):
        for row in range(n):
            if row != state[col]:
                nb = list(state)
                nb[col] = row
                result.append(nb)
    return result

In [8]:
# Pruebas de la actividad 3
test_state = [0, 1, 2, 3]
test_neighbors = neighbors(test_state)

assert len(test_neighbors) == 12
assert test_state not in test_neighbors
assert len({tuple(s) for s in test_neighbors}) == 12

print("Pruebas superadas.")

Pruebas superadas.


## Actividad 4. Mejor vecino

Implemente una función que encuentre el menor costo entre todos los vecinos.

Cuando existan varios vecinos con el mismo costo mínimo, seleccione uno aleatoriamente.

In [9]:
def best_neighbor(state):
    """Un vecino de costo mínimo (desempate aleatorio entre los mejores) y su costo."""
    nbs = neighbors(state)
    costs = [cost(s) for s in nbs]
    m = min(costs)
    best = [s for s, c in zip(nbs, costs) if c == m]
    return random.choice(best), m

In [10]:
# Prueba básica
state = [0, 0, 0, 0]
next_state, next_cost = best_neighbor(state)

assert next_state in neighbors(state)
assert next_cost == cost(next_state)
assert next_cost == min(cost(s) for s in neighbors(state))

print("Pruebas superadas.")

Pruebas superadas.


## 4. Hill Climbing

Hill Climbing comienza en un estado inicial y se mueve repetidamente hacia un vecino con menor costo.

$$
x_{t+1} = \arg\min_{y \in N(x_t)} C(y)
$$

El algoritmo termina cuando:

- encuentra una solución con costo cero; o
- ningún vecino mejora el estado actual.

Este segundo caso puede corresponder a un **mínimo local** o a una **meseta**.

## Actividad 5. Implementar Hill Climbing

In [11]:
def hill_climbing(initial_state, max_steps=100):
    """
    Hill Climbing: sube al mejor vecino solo si mejora ESTRICTAMENTE.

    Retorna:
        final_state: estado alcanzado
        history: lista de estados visitados
    """
    state = list(initial_state)
    history = [list(state)]
    for _ in range(max_steps):
        cand, cand_cost = best_neighbor(state)
        if cand_cost < cost(state):
            state = cand
            history.append(list(state))
        else:
            break
    return state, history

In [12]:
initial_state = [0, 0, 0, 0]
final_state, history = hill_climbing(initial_state)

print("Estado inicial:", initial_state)
print("Estado final:", final_state)
print("Costo final:", cost(final_state))
print("Número de movimientos:", len(history) - 1)

plot_board(initial_state, "Estado inicial")
plot_board(final_state, f"Estado final — costo {cost(final_state)}")

Estado inicial: [0, 0, 0, 0]
Estado final: [1, 3, 0, 2]
Costo final: 0
Número de movimientos: 3


## Actividad 6. Analizar una ejecución

Grafique el costo de los estados visitados por Hill Climbing.

In [13]:
def plot_cost_history(history, title="Evolución del costo"):
    costs = [cost(state) for state in history]

    plt.figure(figsize=(7, 4))
    plt.plot(range(len(costs)), costs, marker="o")
    plt.xlabel("Iteración")
    plt.ylabel("Costo")
    plt.title(title)
    plt.xticks(range(len(costs)))
    plt.show()


plot_cost_history(history)

/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_33715/3416362969.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Hill Climbing con Random Restart

Hill Climbing puede quedar atrapado en mínimos locales.

Random Restart ejecuta Hill Climbing desde diferentes estados iniciales:

$$
x_0^{(1)}, x_0^{(2)}, \ldots, x_0^{(R)}
$$

El algoritmo termina cuando encuentra una solución o cuando agota el número máximo de reinicios.

## Actividad 7. Implementar Random Restart

In [14]:
def random_state(n=N):
    """Genera un estado aleatorio."""
    return [random.randrange(n) for _ in range(n)]


def random_restart_hill_climbing(max_restarts=50, max_steps=100):
    """Hill Climbing desde varios estados aleatorios; conserva la mejor ejecución.

    Retorna: best_state, best_history, restarts_used
    """
    best_state, best_history = None, None
    for restart in range(1, max_restarts + 1):
        state, history = hill_climbing(random_state(), max_steps)
        if best_state is None or cost(state) < cost(best_state):
            best_state, best_history = state, history
        if cost(best_state) == 0:
            return best_state, best_history, restart
    return best_state, best_history, max_restarts

In [15]:
best_state, best_history, restarts = random_restart_hill_climbing()

print("Mejor estado:", best_state)
print("Costo:", cost(best_state))
print("Reinicios utilizados:", restarts)

plot_board(best_state, f"Random Restart — costo {cost(best_state)}")
plot_cost_history(best_history, "Mejor ejecución de Random Restart")

Mejor estado: [2, 0, 3, 1]
Costo: 0
Reinicios utilizados: 4


/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_33715/3416362969.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Simulated Annealing

Simulated Annealing permite aceptar temporalmente movimientos que empeoran la solución.

Si un vecino tiene costo menor o igual, se acepta.

Si empeora el costo en una cantidad:

$$
\Delta C = C(x') - C(x) > 0
$$

se acepta con probabilidad:

$$
P(\text{aceptar}) = e^{-\Delta C/T}
$$

donde $T$ es la temperatura.

Usaremos un enfriamiento geométrico:

$$
T_{t+1} = \alpha T_t
$$

con $0 < \alpha < 1$.

## Actividad 8. Probabilidad de aceptación

In [16]:
def acceptance_probability(current_cost, candidate_cost, temperature):
    """Minimización: 1.0 si el candidato mejora o iguala; exp(-delta/T) si empeora."""
    delta = candidate_cost - current_cost
    if delta <= 0:
        return 1.0
    if temperature <= 0:
        return 0.0
    return math.exp(-delta / temperature)

In [17]:
assert acceptance_probability(3, 2, 1.0) == 1.0
assert acceptance_probability(3, 3, 1.0) == 1.0

p = acceptance_probability(2, 3, 1.0)
assert math.isclose(p, math.exp(-1), rel_tol=1e-9)

print("Pruebas superadas.")

Pruebas superadas.


## Actividad 9. Implementar Simulated Annealing

En cada iteración:

1. seleccione un vecino aleatorio;
2. calcule el cambio de costo;
3. acepte siempre las mejoras;
4. acepte algunos movimientos peores según la temperatura;
5. reduzca la temperatura.

In [18]:
def simulated_annealing(
    initial_state,
    initial_temperature=10.0,
    cooling_rate=0.95,
    min_temperature=1e-3,
    max_steps=1000,
):
    """
    Simulated Annealing para N reinas.

    Retorna: best_state, history (estados aceptados), temperatures
    """
    state = list(initial_state)
    current_cost = cost(state)
    best_state, best_cost = list(state), current_cost
    history = [list(state)]
    temperatures = [initial_temperature]
    T = initial_temperature

    for _ in range(max_steps):
        if T < min_temperature or current_cost == 0:
            break
        candidate = random.choice(neighbors(state))
        candidate_cost = cost(candidate)
        if random.random() < acceptance_probability(current_cost, candidate_cost, T):
            state, current_cost = candidate, candidate_cost
            history.append(list(state))
            temperatures.append(T)
            if current_cost < best_cost:
                best_state, best_cost = list(state), current_cost
        T *= cooling_rate

    return best_state, history, temperatures

In [19]:
initial_state = random_state()
sa_state, sa_history, temperatures = simulated_annealing(initial_state)

print("Estado inicial:", initial_state)
print("Mejor estado:", sa_state)
print("Costo final:", cost(sa_state))
print("Estados aceptados:", len(sa_history))

plot_board(initial_state, "Estado inicial")
plot_board(sa_state, f"Simulated Annealing — costo {cost(sa_state)}")
plot_cost_history(sa_history, "Evolución del costo en Simulated Annealing")

Estado inicial: [3, 0, 2, 2]
Mejor estado: [2, 0, 3, 1]
Costo final: 0
Estados aceptados: 53


/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_33715/3416362969.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Actividad 10. Comparación experimental

Ejecute cada algoritmo varias veces y registre su tasa de éxito.

Una ejecución es exitosa cuando encuentra un estado con costo cero.

In [20]:
def compare_algorithms(trials=100):
    results = {"Hill Climbing": 0, "Random Restart": 0, "Simulated Annealing": 0}
    for _ in range(trials):
        initial_state = random_state()
        hc_state, _ = hill_climbing(initial_state)
        if cost(hc_state) == 0:
            results["Hill Climbing"] += 1
        rr_state, _, _ = random_restart_hill_climbing(max_restarts=20)
        if cost(rr_state) == 0:
            results["Random Restart"] += 1
        sa_state, _, _ = simulated_annealing(initial_state)
        if cost(sa_state) == 0:
            results["Simulated Annealing"] += 1
    return results


results = compare_algorithms(trials=100)
for algorithm, successes in results.items():
    print(f"{algorithm:20s}: {successes}/100")
# Con solo 4 reinas los tres resuelven casi siempre; Random Restart y Simulated
# Annealing evitan quedarse atrapados en los pocos óptimos locales del problema.

Hill Climbing       : 36/100
Random Restart      : 100/100
Simulated Annealing : 98/100


## Preguntas de discusión

1. ¿Por qué Hill Climbing puede fallar aunque exista una solución?
2. ¿Qué diferencia existe entre una meseta y un mínimo local?
3. ¿Por qué Random Restart mejora la probabilidad de encontrar una solución?
4. ¿Qué ocurre en Simulated Annealing cuando la temperatura es muy alta?
5. ¿Qué ocurre si la temperatura disminuye demasiado rápido?
6. ¿Cómo cambiaría el tamaño del espacio de búsqueda para 8 reinas?
7. ¿Cuál de los tres algoritmos fue más confiable en los experimentos?

## Reto adicional

Modifique:

```python
N = 8
```

y evalúe los algoritmos en el problema de las **8 reinas**.

Compare:

- tasa de éxito;
- número de iteraciones;
- número de reinicios;
- sensibilidad a la temperatura inicial;
- sensibilidad a la tasa de enfriamiento.